### Day 3 Assignment: CTAS, read_files(), Metadata Columns & Iceberg

### Basic Tasks

#### 1. CTAS with read_files()

In [0]:
create table dev.demo.circuits as
select * from read_files("/Volumes/dev/demo/ex-volume/circuits.csv")

#### 2. Ingest a nested JSON file

In [0]:
create table dev.demo.drivers_table as
select * from read_files("/Volumes/dev/demo/ex-volume/drivers.json")

In [0]:
select * except (name, `_rescued_data`), name.forename, name.surname from dev.demo.drivers_table

#### 3. DESCRIBE statements

In [0]:
DESCRIBE dev.demo.drivers_table

In [0]:
DESCRIBE EXTENDED dev.demo.drivers_table

In [0]:
DESCRIBE DETAIL dev.demo.drivers_table

### Intermediate Tasks

#### 4. Add metadata columns

In [0]:
select *, _metadata.file_name, _metadata.file_path from dev.demo.drivers_table

#### 5. Create an Iceberg table

In [0]:
create table if not exists dev.demo.drivers_table_2 using iceberg as
select * from read_files("/Volumes/dev/demo/ex-volume/drivers.json") 

In [0]:
DESCRIBE DETAIL dev.demo.drivers_table_2

#### 6. Records per source file

In [0]:
select
 _metadata.file_name,
 _metadata.file_path,
 _metadata.file_modification_time,
 count(*) as record_count
from dev.demo.drivers_table
group by _metadata.file_name, _metadata.file_path, _metadata.file_modification_time
order by record_count desc

### Advanced Tasks 


#### 7. Multiple CSV files

In [0]:
 -- 1. Standard CSV with header and default comma delimiter
CREATE OR REPLACE TABLE dev.demo.ingest_standard_csv AS
SELECT * 
FROM read_files(
  "/Volumes/dev/demo/ex-volume/standard/circuits.csv",
  format => "csv",
  header => "true",
  inferSchema => "true"
);

In [0]:
-- 2. Pipe-delimited CSV with custom encoding and quotes
CREATE OR REPLACE TABLE dev.demo.ingest_piped_csv AS
SELECT * 
FROM read_files(
  "/Volumes/dev/demo/ex-volume/piped/*.csv",
  format => "csv",
  header => "true",
  sep => "|",
  quote => "\"",
  escape => "\\"
);
SELECT * FROM dev.demo.ingest_piped_csv LIMIT 5;

In [0]:
-- 3. Handling Schema Evolution (e.g., extra columns added by vendor)
CREATE OR REPLACE TABLE dev.demo.ingest_evolving_csv AS
SELECT * 
FROM read_files(
  "/Volumes/dev/demo/ex-volume/evolving/*.csv",
  format => "csv",
  header => "true",
  schema => "driverId INT, code STRING, forename STRING, surname STRING",
  -- schemaEvolutionMode => "addNewColumns", --Schema evolution mode addNewColumns is not supported when the schema is specified.
  rescuedDataColumn => "_rescued_data"
);
select * from dev.demo.ingest_evolving_csv

#### 8. Storage Format Selection: Native Delta vs. Iceberg / Delta UniForm


As we expand our analytical ecosystem, our data is no longer consumed solely within Databricks. Downstream query engines—primarily **Snowflake** and **Trino**—need low-latency, governed access to the same tables without creating redundant data pipelines or duplicate storage costs.


**When to Use Delta UniForm (Our Recommended Default)**

Choose Delta UniForm for all shared gold and silver reporting tables that downstream tools like Snowflake or Trino need to query.

* Why: UniForm writes data once as standard Parquet files with Delta commits, but automatically generates Iceberg metadata in the background.
* The Benefit: Databricks keeps peak write speeds and features (like Liquid Clustering), while Snowflake and Trino can read the table directly as an Apache Iceberg table through Unity Catalog with zero data copying.


**When to Use Native Delta Lake**
Choose Native Delta for tables that live and stay strictly inside Databricks.

* Best For: Bronze landing zones, high-throughput streaming jobs (Auto Loader / Kafka), intermediate feature stores, and Databricks-only BI dashboards.
* Why: It avoids the minor overhead of generating Iceberg metadata where external engines will never read the data.


**When to Use Native Apache Iceberg**
Choose Native Iceberg only when non-Databricks engines require direct write access.

* Best For: Multi-cloud pipelines where engines like Trino or Apache Flink write directly into the table without routing through Databricks compute.


**In-Short**

* **Databricks-only ETL / Landing:** Keep Native Delta.
* **Databricks writes, Snowflake/Trino reads:** Enable Delta UniForm.
* **Trino/Flink writes directly:** Use Native Iceberg.

#### 9. DESCRIBE HISTORY 

In [0]:
SELECT 
    *, 
    _rescued_data AS corrupted_record,
    _metadata.file_name AS corrupted_file_name,
    _metadata.file_path AS corrupted_file_path,
    _metadata.file_modification_time AS file_drop_timestamp
FROM dev.demo.ingest_evolving_csv
WHERE driverId IS NULL OR _rescued_data IS NOT NULL;

In [0]:
DESCRIBE HISTORY dev.demo.ingest_evolving_csv

In [0]:
-- Did it exist in version 0?
SELECT count(*) FROM dev.demo.ingest_evolving_csv VERSION AS OF 0 
WHERE _rescued_data IS NOT NULL;

In [0]:
-- It existed in version 11?
-- We guessed it by looking at the timestamp in the DESCRIBE HISTORY command above
SELECT count(*) FROM dev.demo.ingest_evolving_csv VERSION AS OF 11
WHERE _rescued_data IS NOT NULL;